In [1]:
# Importing modules
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# Reading data
df_ml = pd.read_csv("results/Output_ML_Analysis.csv")
df_ml["Type"] = "ML"
df_gnn = pd.read_csv("results/Output_GNN_Analysis.csv")
df_gnn["Type"] = "GNN"
df_gnnfp = pd.read_csv("results/Output_GNN_FP_Analysis.csv")
df_gnnfp["Model"] = [i+"+FP" for i in df_gnnfp["Model"]]
df_gnnfp["Type"] = "GNN_FP"

In [3]:
# Merging data
merged_df = pd.concat([df_ml, df_gnn, df_gnnfp]).reset_index(drop=True)

In [4]:
# Function to plot barplot showing result
def plot_bar(data, target):
    data = data.copy()
    data["err_lower"] = data[target] - data[f"{target}_lower"]
    data["err_upper"] = data[f"{target}_upper"] - data[target]
    sns.set_theme(style="ticks", context="paper")
    model_order = data["Model"].unique()
    g = sns.catplot(
        data=data, kind="bar", x="Model", y=target, hue="Model",
        col="Data", col_wrap=2, sharey=False,sharex=False, height=5, width=0.8, 
        aspect=1, dodge=False, errorbar=None, order=model_order)
    for i, ax in enumerate(g.axes.flat):
        col_name = g.col_names[i]
        subdata = data[data["Data"] == col_name].set_index("Model").reindex(model_order).reset_index()
        if not subdata.dropna(subset=[target]).empty:
            ax.errorbar(
                x=np.arange(len(subdata)), 
                y=subdata[target],
                yerr=[subdata["err_lower"], subdata["err_upper"]], 
                fmt="none", capsize=4, linewidth=1, color="black")
    g.set_axis_labels("Model", f"{target}")
    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=90)
        ax.grid(True)
    plt.tight_layout()
    plt.savefig(f"results/{target}_Plot.png", dpi=300)
    plt.close()

In [5]:
# Plotting barplot for RMSE
plot_bar(merged_df, "RMSE")